# Agents, runtime, timers, and messages

Control agents are event targets. They do not run by themselves; `SessionRuntime` binds them, schedules start events, and gives each hook an event-local context.


In [ ]:
from dataclasses import dataclass, field

from simyuj.components import connect_ports
from simyuj.control import AGENT_MESSAGE, AGENT_REPORT, Agent, AgentContext, SessionRuntime
from simyuj.control.payloads import AgentMessage, AgentReport, AgentStart, TimerFired
from simyuj.engine import Event, Timeline
from simyuj.network import Network, Node
from simyuj.primitives.messages import ClassicalMessage


Make two agents. Alice starts a timer, then sends a classical ping. Bob answers with a pong.


In [ ]:
def send_ping(agent: Agent, ctx: AgentContext) -> None:
    message = ClassicalMessage(
        sender_id="alice",
        receiver_id="bob",
        body="ping: round 1",
        sent_time=ctx.timeline.current_time,
        message_id="message:ping:1",
    )
    ctx.classical.send(message, ctx.timeline)
    agent.log.append("scheduled ping to bob")


def send_pong(agent: Agent, ctx: AgentContext) -> None:
    reply = ClassicalMessage(
        sender_id="bob",
        receiver_id="alice",
        body="pong: round 1",
        sent_time=ctx.timeline.current_time,
        message_id="message:pong:1",
    )
    ctx.classical.send(reply, ctx.timeline)
    agent.log.append("scheduled pong to alice")


In [ ]:
@dataclass(slots=True)
class ChatAgent(Agent):
    log: list[str] = field(default_factory=list)

    def on_start(self, start: AgentStart, ctx: AgentContext) -> None:
        self.log.append(f"t={ctx.timeline.current_time}: start on {ctx.node_id}")
        if self.agent_id == "alice":
            ctx.timers.set("send-ping", 2, correlation_id="round-1")

    def on_timer(self, timer: TimerFired, ctx: AgentContext) -> None:
        self.log.append(f"t={ctx.timeline.current_time}: timer {timer.timer_id}")
        if self.agent_id == "alice" and timer.timer_id == "send-ping":
            send_ping(self, ctx)

    def on_message(self, message: AgentMessage, ctx: AgentContext) -> None:
        body = message.message.body
        self.log.append(f"t={ctx.timeline.current_time}: message {body}")
        if self.agent_id == "bob" and body.startswith("ping"):
            send_pong(self, ctx)

    def on_report(self, report: object, ctx: AgentContext) -> None:
        self.log.append(f"t={ctx.timeline.current_time}: report {report}")


In [ ]:
alice_agent = ChatAgent(agent_id="alice")
bob_agent = ChatAgent(agent_id="bob")

alice_endpoint = alice_agent.enable_classical()
bob_endpoint = bob_agent.enable_classical()

alice_endpoint.add_route("bob", "to_bob")
bob_endpoint.add_route("alice", "to_alice")

print("Alice classical address:", alice_endpoint.address)
print("Bob classical address:", bob_endpoint.address)


Classical endpoints own ports. The endpoints are adapters; the agents are the event targets.


In [ ]:
connect_ports(
    alice_endpoint.out_port("to_bob"),
    bob_endpoint.in_port("from_alice"),
    target_action=AGENT_MESSAGE,
)
connect_ports(
    bob_endpoint.out_port("to_alice"),
    alice_endpoint.in_port("from_bob"),
    target_action=AGENT_MESSAGE,
)

print("Alice output port owner:", alice_endpoint.out_port("to_bob").owner_id)
print("Bob input port owner:", bob_endpoint.in_port("from_alice").owner_id)


In [ ]:
timeline = Timeline(master_seed=11)
network = Network("chat_control")

alice_node = Node("alice_node")
bob_node = Node("bob_node")
alice_node.add_agent(alice_agent)
bob_node.add_agent(bob_agent)
network.add_node(bob_node)
network.add_node(alice_node)

runtime = SessionRuntime(
    timeline=timeline,
    network=network,
    session_id="chat-session",
)

print("Runtime agent order:", [agent.agent_id for agent in runtime.agents])


Running the runtime schedules starts, timers, and classical deliveries through the timeline.


In [ ]:
runtime.run()

print("Timeline current time:", timeline.current_time)
print("Events scheduled:", timeline.events_scheduled)
print("Events executed:", timeline.events_executed)


In [ ]:
print("Alice log:")
for item in alice_agent.log:
    print(" ", item)

print("Bob log:")
for item in bob_agent.log:
    print(" ", item)


Reports use the same agent event path. Here we schedule one direct report after the chat.


In [ ]:
report_event = timeline.schedule(
    Event(
        time=timeline.current_time + 1,
        target_ref=alice_agent,
        action=AGENT_REPORT,
        payload_ref=AgentReport(report={"detector": "left", "clicks": 1}),
        source="notebook",
        subsystem_id="control",
    )
)

print("Scheduled report event:", report_event.event_id, "at", report_event.time)


In [ ]:
runtime.run_until_empty()

print("Alice log after report:")
for item in alice_agent.log:
    print(" ", item)


Timer choices are explicit: replace cancels the old event; set-once keeps it.


In [ ]:
@dataclass(slots=True)
class DeadlineAgent(Agent):
    mode: str = "replace"
    fired: list[tuple[str, int]] = field(default_factory=list)

    def on_start(self, start: AgentStart, ctx: AgentContext) -> None:
        ctx.timers.at("deadline", 10)
        if self.mode == "replace":
            ctx.timers.at("deadline", 20, replace=True)
        if self.mode == "set_once":
            ctx.timers.at("deadline", 20, set_once=True)
        if self.mode == "cancel":
            ctx.timers.cancel("deadline")

    def on_timer(self, timer: TimerFired, ctx: AgentContext) -> None:
        self.fired.append((timer.timer_id, ctx.timeline.current_time))


In [ ]:
def run_deadline_mode(mode):
    local_timeline = Timeline(master_seed=3)
    local_network = Network(f"deadline_{mode}")
    node = Node("controller")
    agent = DeadlineAgent(agent_id=f"agent-{mode}", mode=mode)
    node.add_agent(agent)
    local_network.add_node(node)
    SessionRuntime(local_timeline, local_network, session_id=f"session-{mode}").run()
    return agent.fired

for mode in ("replace", "set_once", "cancel"):
    print(mode, "->", run_deadline_mode(mode))


The rule of thumb: an agent hook decides what to schedule next; the timeline decides when it runs.
